# Naturvårdsverket – Friluftsliv: leder, anordningar och publiceringsstatus

Den här notebooken hämtar de fyra aktuella GeoJSON-filerna från Naturvårdsverkets öppna data och gör en gemensam metadata-/identifieringsanalys.

**Datasets**
- `Leder.geojson`
- `Anordningar.geojson`
- `Publiceringsstatus.geojson`
- `Statliga_Leder.geojson`

Notebooken:
1. läser in hela datamängden,
2. bevarar **alla attributfält** och även hela `properties` som JSON,
3. hanterar källsystemet **SWEREF 99 TM (EPSG:3006)** och skapar WGS84-koordinater,
4. letar efter länkar/identifierare till **OpenStreetMap** och **Wikidata**,
5. skapar OSM-länkar – med objektlänk om OSM-ID finns, annars en OSM-kartlänk från koordinaten,
6. identifierar fält för typ/undertyp/kategori och räknar förekomster,
7. skapar sammanfattande DataFrames för varje dataset och en gemensam översikt.

Naturvårdsverkets dokumentation anger att data lagras/tillhandahålls i SWEREF 99 TM och att EPSG:4326 (WGS84) är ett av de tillgängliga referenssystemen via tjänsterna. Källa: https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Leder_och_friluftsanordningar_beskrivning_av_oppna_data.pdf

> **Obs:** Notebooken försöker upptäcka OSM/Wikidata dynamiskt i fältnamn och värden. Det gör att den också fångar kopplingar även om Naturvårdsverket ändrar namngivningen på ett attribut i framtiden.


In [1]:
# 1. Installation – kör endast om paket saknas
# !pip install geopandas pyogrio shapely pandas requests tqdm openpyxl folium

import io
import json
import re
import warnings
from pathlib import Path

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


In [2]:
# 2. Källor och grundinställningar

BASE_URL = "https://geodata.naturvardsverket.se/nedladdning/friluftsliv/"

SOURCES = {
    "leder": BASE_URL + "Leder.geojson",
    "anordningar": BASE_URL + "Anordningar.geojson",
    "publiceringsstatus": BASE_URL + "Publiceringsstatus.geojson",
    "statliga_leder": BASE_URL + "Statliga_Leder.geojson",
}

# Naturvårdsverkets dokumentation anger SWEREF 99 TM.
SOURCE_CRS = "EPSG:3006"
TARGET_CRS = "EPSG:4326"

DATA_DIR = Path("naturvardsverket_friluftsliv_data")
DATA_DIR.mkdir(exist_ok=True)

print("Källor:")
for name, url in SOURCES.items():
    print(f"  {name:22} {url}")


Källor:
  leder                  https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Leder.geojson
  anordningar            https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Anordningar.geojson
  publiceringsstatus     https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Publiceringsstatus.geojson
  statliga_leder         https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Statliga_Leder.geojson


In [3]:
# 3. Hjälpfunktioner för nedladdning och läsning

def download_geojson(url, local_path):
    """Ladda ner GeoJSON och spara lokalt. Returnerar sökvägen."""
    local_path = Path(local_path)
    if local_path.exists() and local_path.stat().st_size > 0:
        print(f"Finns redan: {local_path} ({local_path.stat().st_size/1e6:.1f} MB)")
        return local_path

    print(f"Laddar ner {url}")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    local_path.write_bytes(r.content)
    print(f"Sparad: {local_path} ({len(r.content)/1e6:.1f} MB)")
    return local_path


def read_dataset(name, url):
    path = download_geojson(url, DATA_DIR / f"{name}.geojson")

    # pyogrio/fiona kan läsa GeoJSON direkt från fil.
    gdf = gpd.read_file(path)

    # GeoJSON-filerna kan sakna CRS-information trots att data ligger i SWEREF 99 TM.
    if gdf.crs is None:
        gdf = gdf.set_crs(SOURCE_CRS, allow_override=True)
    return gdf


datasets = {}
for name, url in SOURCES.items():
    datasets[name] = read_dataset(name, url)

for name, gdf in datasets.items():
    print(f"{name:22} {len(gdf):>8,} features | CRS: {gdf.crs} | geometrier: {gdf.geom_type.value_counts().to_dict()}")


Laddar ner https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Leder.geojson
Sparad: naturvardsverket_friluftsliv_data/leder.geojson (51.4 MB)
Laddar ner https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Anordningar.geojson
Sparad: naturvardsverket_friluftsliv_data/anordningar.geojson (13.1 MB)
Laddar ner https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Publiceringsstatus.geojson
Sparad: naturvardsverket_friluftsliv_data/publiceringsstatus.geojson (88.4 MB)
Laddar ner https://geodata.naturvardsverket.se/nedladdning/friluftsliv/Statliga_Leder.geojson
Sparad: naturvardsverket_friluftsliv_data/statliga_leder.geojson (21.7 MB)
leder                    12,011 features | CRS: EPSG:3006 | geometrier: {'LineString': 11976, 'MultiLineString': 35}
anordningar              21,579 features | CRS: EPSG:3006 | geometrier: {'Point': 21579}
publiceringsstatus        8,790 features | CRS: EPSG:3006 | geometrier: {'Polygon': 7559, 'MultiPolygon': 1231}
statliga_leder        

## 4. Metadataanalys

Vi skiljer här på:

- **attributmetadata** – alla fält i `properties`,
- **geometri** – typ, längd/area där det är meningsfullt,
- **identifierare** – fält som verkar innehålla OSM/Wikidata/URL/ID,
- **typologi** – typ, undertyp och kategori,
- **WGS84** – longitud/latitud för en representativ punkt.

Alla ursprungliga attribut sparas i resultatet. Dessutom sparas `metadata_json`, så att inget attribut behöver gå förlorat om vi senare ändrar vilka kolumner vi analyserar.


In [4]:
# 5. Generella funktioner för identifiering av OSM/Wikidata och typfält

URL_RE = re.compile(r"https?://[^\s\"']+", re.I)
WIKIDATA_Q_RE = re.compile(r"(?<![A-Za-z0-9_])(Q\d+)(?![A-Za-z0-9_])", re.I)
WIKIDATA_URL_RE = re.compile(r"https?://(?:www\.)?wikidata\.org/entity/(Q\d+)", re.I)
OSM_URL_RE = re.compile(r"https?://(?:www\.)?openstreetmap\.org/[^\s\"']+", re.I)

OSM_KEY_RE = re.compile(r"(?:^|[_:.-])(osm|openstreetmap)(?:$|[_:.-])", re.I)
WIKIDATA_KEY_RE = re.compile(r"wikidata", re.I)
TYPE_KEY_RE = re.compile(
    r"(typ|type|undertyp|subtyp|kategori|category|class|klass|slag)",
    re.I
)

def stringify(value):
    if pd.isna(value) if not isinstance(value, (list, dict)) else False:
        return ""
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

def extract_wikidata(value):
    s = stringify(value)
    m = WIKIDATA_URL_RE.search(s)
    if m:
        return m.group(1).upper()
    m = WIKIDATA_Q_RE.search(s)
    return m.group(1).upper() if m else None

def extract_osm_url(value):
    s = stringify(value)
    m = OSM_URL_RE.search(s)
    return m.group(0).rstrip(".,);]") if m else None

def find_identifier_columns(gdf):
    cols = []
    for c in gdf.columns:
        if c == "geometry":
            continue
        cstr = str(c)
        if OSM_KEY_RE.search(cstr) or WIKIDATA_KEY_RE.search(cstr):
            cols.append(c)
    return cols

def find_type_columns(gdf):
    return [
        c for c in gdf.columns
        if c != "geometry" and TYPE_KEY_RE.search(str(c))
    ]

def find_url_columns(gdf):
    cols = []
    for c in gdf.columns:
        if c == "geometry":
            continue
        sample = gdf[c].dropna().astype(str).head(200)
        if sample.str.contains(r"^https?://", regex=True, case=False).any():
            cols.append(c)
    return cols

def metadata_schema(gdf):
    rows = []
    for c in gdf.columns:
        if c == "geometry":
            continue
        s = gdf[c]
        rows.append({
            "field": c,
            "dtype": str(s.dtype),
            "non_null": int(s.notna().sum()),
            "null": int(s.isna().sum()),
            "unique": int(s.nunique(dropna=True)),
            "sample": " | ".join(map(str, s.dropna().head(3).tolist()))
        })
    return pd.DataFrame(rows).sort_values(["non_null", "field"], ascending=[False, True])


In [5]:
# 6. Skapa ett analyserat DataFrame för ett dataset

def analyse_dataset(name, gdf):
    out = gdf.copy()

    # Behåll exakt alla ursprungliga properties som JSON.
    property_cols = [c for c in out.columns if c != "geometry"]
    out["metadata_json"] = out[property_cols].apply(
        lambda row: json.dumps(
            {k: (None if pd.isna(v) else v) for k, v in row.items()},
            ensure_ascii=False,
            default=str
        ),
        axis=1
    )

    # Käll-CRS -> WGS84
    if out.crs is None:
        out = out.set_crs(SOURCE_CRS, allow_override=True)
    wgs = out.to_crs(TARGET_CRS)

    # Representativ punkt: point = sig själv, linje/polygon = centroid.
    # Centroiden används endast som koordinat för länkar/analys – geometrin ändras inte.
    representative = wgs.geometry.centroid

    out["longitude_wgs84"] = representative.x
    out["latitude_wgs84"] = representative.y
    out["geometry_type"] = out.geometry.geom_type.astype(str)

    # Längd/area i meter/m² beräknas i EPSG:3006.
    try:
        out["length_m"] = out.geometry.length
    except Exception:
        out["length_m"] = None

    try:
        out["area_m2"] = out.geometry.area
    except Exception:
        out["area_m2"] = None

    # Identifieringsfält
    id_cols = find_identifier_columns(out)
    url_cols = find_url_columns(out)

    def row_values(row):
        vals = []
        for c in property_cols:
            v = row.get(c)
            if pd.notna(v):
                vals.append((c, stringify(v)))
        return vals

    osm_links = []
    wikidata_links = []
    osm_sources = []
    wikidata_ids = []

    for _, row in out.iterrows():
        pairs = row_values(row)

        osm_url = None
        osm_source = None
        for c, v in pairs:
            u = extract_osm_url(v)
            if u:
                osm_url = u
                osm_source = c
                break

        # Leta efter OSM-ID + eventuell typ.
        if not osm_url:
            osm_id = None
            osm_type = None
            for c, v in pairs:
                if OSM_KEY_RE.search(str(c)):
                    if re.fullmatch(r"\d+", v.strip()):
                        osm_id = v.strip()
                    if "type" in str(c).lower() and v.lower() in {"node", "way", "relation"}:
                        osm_type = v.lower()
            if osm_id and osm_type:
                osm_url = f"https://www.openstreetmap.org/{osm_type}/{osm_id}"
                osm_source = "constructed_from_osm_id"

        # Fallback: OSM-kartlänk på representativ koordinat.
        if not osm_url:
            lon = row["longitude_wgs84"]
            lat = row["latitude_wgs84"]
            if pd.notna(lon) and pd.notna(lat):
                osm_url = f"https://www.openstreetmap.org/?mlat={lat:.7f}&mlon={lon:.7f}#map=18/{lat:.7f}/{lon:.7f}"
                osm_source = "coordinate_fallback"

        osm_links.append(osm_url)
        osm_sources.append(osm_source)

        qid = None
        for c, v in pairs:
            qid = extract_wikidata(v)
            if qid:
                break
        wikidata_ids.append(qid)
        wikidata_links.append(
            f"https://www.wikidata.org/wiki/{qid}" if qid else None
        )

    out["osm_link"] = osm_links
    out["osm_link_source"] = osm_sources
    out["wikidata_id"] = wikidata_ids
    out["wikidata_link"] = wikidata_links
    out["dataset"] = name

    return out


analysed = {
    name: analyse_dataset(name, gdf)
    for name, gdf in datasets.items()
}

for name, df in analysed.items():
    print(f"{name}: {len(df):,} rader, {len(df.columns):,} kolumner")


leder: 12,011 rader, 26 kolumner
anordningar: 21,579 rader, 25 kolumner
publiceringsstatus: 8,790 rader, 15 kolumner
statliga_leder: 2,004 rader, 16 kolumner


In [6]:
# 7. Visa ALLA metadatafält för respektive dataset

metadata_tables = {
    name: metadata_schema(gdf)
    for name, gdf in datasets.items()
}

for name, table in metadata_tables.items():
    print("\n" + "="*90)
    print(name.upper())
    display(table)



LEDER


,field,dtype,non_null,null,unique,sample
1,GEOMETRIKVALITET,object,12011,0,6,Okänd noggrannhet | Okänd noggrannhet | >20-50 meter
7,LEDKATEGORI,object,12011,0,3,Barmarksled | Barmarksled | Barmarksled
12,LEDLANGD,int32,12011,0,2796,933 | 4039 | 4672
6,LEDTYP,object,12011,0,32,Vandringsled | Vandringsled | Vandringsled
2,LED_ID,object,12011,0,3655,30207711 | 30208094 | 30207434
0,OBJECTID,int32,12011,0,11669,12347649 | 12345319 | 12348897
13,SHAPE.LEN,float64,12011,0,11086,933.252884865883 | 4038.95449174816 | 259.40319261292
10,SKYDDATOMRADE,object,10488,1523,1932,Inre Kilsviken (2002209) | Åbengtshöjden (2002227) | Högbergsfältet (2002137)
11,SKYDDATOMRADE_ID,object,10488,1523,1932,2002209 | 2002227 | 2002137
3,LEDNAMN,object,6959,5052,1802,Led Hygntornet | Led Åbengtshöjden | Led Högbergsfältet



ANORDNINGAR


,field,dtype,non_null,null,unique,sample
1,ANORDNING_ID,object,21579,0,21550,30335917 | 30474194 | 30467932
6,NP,int32,21579,0,2,1 | 0 | 0
0,OBJECTID,int32,21579,0,21579,33776721 | 33788626 | 33781407
3,TYP,object,21579,0,61,Fyr | Fyr | Fyr
4,UNDERTYP,object,21579,0,75,Fyr | Fyr | Fyr
7,GEOMETRIKVALITET,object,21474,105,6,<= 20 meter | <= 20 meter | <= 20 meter
9,SKYDDATOMRADE,object,21024,555,4232,"Stenshuvud (2001830) | Kullaberg (SE0430092), Västra Kullaberg (2000972) | Bjärehalvöns kuster (2031810), Hallands V..."
10,SKYDDATOMRADE_ID,object,21024,555,4232,"2001830 | SE0430092, 2000972 | 2031810, 2000962, SE0420002"
8,SKYDDSTYP_KOD,object,21024,555,43,"NP | N2000-SPA/SCI, NR | DVO, N2000-SPA/SCI, NR"
2,ANORDNINGNAMN,object,9562,12017,5707,"Vägbom med kodlås | Stängd vägbom Vålhallberget | Vägbom Tijärnsskogen, Dörrfjället"



PUBLICERINGSSTATUS


,field,dtype,non_null,null,unique,sample
2,DATA_FRILUFTSLIV_PUBLICERAT,object,8790,0,2,Nej | Ja | Nej
1,NAMN,object,8790,0,8235,Kåreholm och Kartarna | Östermoskogen | Idegranar
0,NVRID,object,8790,0,8711,2011415 | 2014384 | 2011931



STATLIGA_LEDER


,field,dtype,non_null,null,unique,sample
0,OBJECTID,int32,2004,0,1921,12348588 | 12348587 | 12348586
3,SHAPE.LEN,float64,2004,0,1792,985.5343687006 | 3766.9228843242 | 2391.08898962359
1,STATLIGLED,object,2004,0,266,Z 59 | Z 59 | Z 59
2,STATLIGLEDNAMN,object,2004,0,266,Vålådalen - Vålåstugorna (justerad) | Vålådalen - Vålåstugorna (justerad) | Vålådalen - Vålåstugorna (justerad)


In [7]:
# 8. Leta specifikt efter OSM, Wikidata och URL-kopplingar

identifier_report = []

for name, gdf in datasets.items():
    for c in gdf.columns:
        if c == "geometry":
            continue

        s = gdf[c].dropna().astype(str)
        if len(s) == 0:
            continue

        osm_hits = int(s.str.contains(r"openstreetmap|osm", case=False, regex=True).sum())
        wikidata_hits = int(s.str.contains(r"wikidata|\\bQ\\d+\\b", case=False, regex=True).sum())
        url_hits = int(s.str.contains(r"^https?://", case=False, regex=True).sum())

        if osm_hits or wikidata_hits or url_hits:
            identifier_report.append({
                "dataset": name,
                "field": c,
                "non_null": len(s),
                "OSM_hits": osm_hits,
                "Wikidata_hits": wikidata_hits,
                "URL_hits": url_hits,
                "unique_values": int(s.nunique())
            })

identifier_report = pd.DataFrame(identifier_report).sort_values(
    ["dataset", "OSM_hits", "Wikidata_hits", "URL_hits"],
    ascending=[True, False, False, False]
)

display(identifier_report)


,dataset,field,non_null,OSM_hits,Wikidata_hits,URL_hits,unique_values
0,anordningar,BESKRIVNING,2632,0,0,1,1870
1,publiceringsstatus,NAMN,8790,1,0,0,8235


In [8]:
# 9. Sammanställ OSM/Wikidata-täckning

coverage_rows = []

for name, df in analysed.items():
    coverage_rows.append({
        "dataset": name,
        "features": len(df),
        "features_with_wikidata": int(df["wikidata_id"].notna().sum()),
        "wikidata_percent": round(100 * df["wikidata_id"].notna().mean(), 2),
        "features_with_explicit_osm_url_or_id": int(
            df["osm_link_source"].isin(["constructed_from_osm_id"]).sum()
        ),
        "features_with_osm_coordinate_link": int(
            df["osm_link"].notna().sum()
        ),
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage)


,dataset,features,features_with_wikidata,wikidata_percent,features_with_explicit_osm_url_or_id,features_with_osm_coordinate_link
0,leder,12011,0,0.0,0,12011
1,anordningar,21579,0,0.0,0,21579
2,publiceringsstatus,8790,0,0.0,0,8790
3,statliga_leder,2004,0,0.0,0,2004


In [9]:
# 10. Typ / undertyp / kategori – automatiskt upptäckta fält

type_reports = {}

for name, gdf in datasets.items():
    fields = find_type_columns(gdf)
    type_reports[name] = fields

    print("\n" + "="*90)
    print(name.upper())
    print("Upptäckta typ-/kategori-/undertypfält:", fields)

    for field in fields:
        counts = (
            gdf[field]
            .fillna("<NULL>")
            .astype(str)
            .value_counts(dropna=False)
            .rename_axis(field)
            .reset_index(name="count")
        )
        print(f"\n{field}")
        display(counts.head(100))



LEDER
Upptäckta typ-/kategori-/undertypfält: ['LEDTYP', 'LEDKATEGORI']

LEDTYP


,LEDTYP,count
0,Vandringsled,9048
1,"Skidled, Skoterled",690
2,Skidled,428
3,Omarkerad stig,343
4,Naturstig,245
5,Skoterled,244
6,"Naturstig, Vandringsled",194
7,"Skidled, Vinterled",164
8,"Skoterled, Vinterled",115
9,Vinterled,113



LEDKATEGORI


,LEDKATEGORI,count
0,Barmarksled,10175
1,Led på snö,1808
2,Led på/i vatten,28



ANORDNINGAR
Upptäckta typ-/kategori-/undertypfält: ['TYP', 'UNDERTYP', 'SKYDDSTYP_KOD']

TYP


...


UNDERTYP


,UNDERTYP,count
0,Områdesskyddsinformation,7135
1,Parkering,3021
2,Bänkbord,1625
3,Eldstad,1571
4,Dass,1080
...,...,...
70,Kajakramp,2
71,Laddningsstation för elbilar,2
72,Barnens skog,1
73,Tillgänglighetsramp,1



SKYDDSTYP_KOD


,SKYDDSTYP_KOD,count
0,NR,16805
1,"N2000-SCI, NR",1282
2,NP,954
3,"N2000-SPA/SCI, NR",799
4,<NULL>,555
5,NVO,313
6,"N2000-SPA/SCI, NP",216
7,"NR, NR",128
8,"N2000-SCI, NP",88
9,DVO,54



PUBLICERINGSSTATUS
Upptäckta typ-/kategori-/undertypfält: []

STATLIGA_LEDER
Upptäckta typ-/kategori-/undertypfält: []


In [10]:
# 11. Kombinerade typologitabeller
# För varje dataset skapas en tabell med kombinationer av alla upptäckta typfält.

type_summary = {}

for name, gdf in datasets.items():
    fields = type_reports[name]

    if fields:
        tmp = gdf[fields].copy().fillna("<NULL>").astype(str)
        summary = (
            tmp.value_counts(dropna=False)
               .reset_index(name="count")
               .sort_values("count", ascending=False)
        )
    else:
        summary = pd.DataFrame({"message": ["Inga uppenbara typfält hittades automatiskt."]})

    type_summary[name] = summary

    print("\n" + "="*90)
    print(f"{name.upper()} – typ/undertyp/kategori")
    display(summary.head(200))



LEDER – typ/undertyp/kategori


,LEDTYP,LEDKATEGORI,count
0,Vandringsled,Barmarksled,9048
1,"Skidled, Skoterled",Led på snö,690
2,Skidled,Led på snö,428
3,Omarkerad stig,Barmarksled,343
4,Naturstig,Barmarksled,245
5,Skoterled,Led på snö,244
6,"Naturstig, Vandringsled",Barmarksled,194
7,"Skidled, Vinterled",Led på snö,164
8,"Skoterled, Vinterled",Led på snö,115
9,Vinterled,Led på snö,113



ANORDNINGAR – typ/undertyp/kategori


,TYP,UNDERTYP,SKYDDSTYP_KOD,count
0,Information,Områdesskyddsinformation,NR,6177
1,Parkering,Parkering,NR,2609
2,Rastplats,Bänkbord,NR,1213
3,Eldstad,Eldstad,NR,1207
4,Dass,Dass,NR,757
...,...,...,...,...
184,Parkering,Parkering,"N2000-SCI, NR, NR",5
183,Parkering,Parkering,"N2000-SCI, NP",5
182,Informationsbyggnad,Informationsbyggnad,NP,5
213,Information,Ledterminal,NR,4



PUBLICERINGSSTATUS – typ/undertyp/kategori


,message
0,Inga uppenbara typfält hittades automatiskt.



STATLIGA_LEDER – typ/undertyp/kategori


,message
0,Inga uppenbara typfält hittades automatiskt.


In [11]:
# 12. Kontroll av datasetens geometri och koordinater

geometry_summary = []

for name, df in analysed.items():
    geometry_summary.append({
        "dataset": name,
        "features": len(df),
        "geometry_types": ", ".join(
            f"{k}: {v}" for k, v in df["geometry_type"].value_counts().items()
        ),
        "min_lon": df["longitude_wgs84"].min(),
        "max_lon": df["longitude_wgs84"].max(),
        "min_lat": df["latitude_wgs84"].min(),
        "max_lat": df["latitude_wgs84"].max(),
    })

geometry_summary = pd.DataFrame(geometry_summary)
display(geometry_summary)


,dataset,features,geometry_types,min_lon,max_lon,min_lat,max_lat
0,leder,12011,"LineString: 11976, MultiLineString: 35",10.995541,23.973842,55.357712,69.052173
1,anordningar,21579,Point: 21579,10.968396,24.049856,55.357107,69.045896
2,publiceringsstatus,8790,"Polygon: 7559, MultiPolygon: 1231",11.008021,24.140104,55.340055,68.541278
3,statliga_leder,2004,LineString: 2004,12.134415,20.954457,61.106628,69.052173


In [12]:
# 13. Exempel: titta på första raderna med alla metadata + WGS84 + länkar

for name, df in analysed.items():
    print("\n" + "="*90)
    print(name.upper())
    cols = [
        c for c in [
            "OBJECTID", "LED_ID", "LEDNAMN", "LEDTYP", "LEDKATEGORI",
            "latitude_wgs84", "longitude_wgs84",
            "osm_link", "wikidata_id", "wikidata_link",
            "geometry_type", "length_m", "area_m2", "metadata_json"
        ] if c in df.columns
    ]
    display(df[cols].head(10))



LEDER


,OBJECTID,LED_ID,LEDNAMN,LEDTYP,LEDKATEGORI,latitude_wgs84,longitude_wgs84,osm_link,wikidata_id,wikidata_link,geometry_type,length_m,area_m2,metadata_json
0,12347649,30207711,Led Hygntornet,Vandringsled,Barmarksled,59.118533,14.086806,https://www.openstreetmap.org/?mlat=59.1185327&mlon=14.0868061#map=18/59.1185327/14.0868061,None,None,LineString,933.252885,0.0,"{""OBJECTID"": 12347649, ""GEOMETRIKVALITET"": ""Okänd noggrannhet"", ""LED_ID"": ""30207711"", ""LEDNAMN"": ""Led Hygntornet"", ""..."
1,12345319,30208094,Led Åbengtshöjden,Vandringsled,Barmarksled,59.813047,14.371673,https://www.openstreetmap.org/?mlat=59.8130470&mlon=14.3716732#map=18/59.8130470/14.3716732,None,None,LineString,4038.954492,0.0,"{""OBJECTID"": 12345319, ""GEOMETRIKVALITET"": ""Okänd noggrannhet"", ""LED_ID"": ""30208094"", ""LEDNAMN"": ""Led Åbengtshöjden""..."
2,12348897,30207434,Led Högbergsfältet,Vandringsled,Barmarksled,59.741454,14.284721,https://www.openstreetmap.org/?mlat=59.7414541&mlon=14.2847208#map=18/59.7414541/14.2847208,None,None,LineString,259.403193,0.0,"{""OBJECTID"": 12348897, ""GEOMETRIKVALITET"": "">20-50 meter"", ""LED_ID"": ""30207434"", ""LEDNAMN"": ""Led Högbergsfältet"", ""B..."
3,12351843,30207434,Led Högbergsfältet,Vandringsled,Barmarksled,59.747851,14.290039,https://www.openstreetmap.org/?mlat=59.7478507&mlon=14.2900393#map=18/59.7478507/14.2900393,None,None,LineString,470.864921,0.0,"{""OBJECTID"": 12351843, ""GEOMETRIKVALITET"": "">20-50 meter"", ""LED_ID"": ""30207434"", ""LEDNAMN"": ""Led Högbergsfältet"", ""B..."
4,12351841,30207434,Led Högbergsfältet,Vandringsled,Barmarksled,59.742122,14.286766,https://www.openstreetmap.org/?mlat=59.7421216&mlon=14.2867658#map=18/59.7421216/14.2867658,None,None,LineString,8.126183,0.0,"{""OBJECTID"": 12351841, ""GEOMETRIKVALITET"": "">20-50 meter"", ""LED_ID"": ""30207434"", ""LEDNAMN"": ""Led Högbergsfältet"", ""B..."
5,12351840,30207434,Led Högbergsfältet,Vandringsled,Barmarksled,59.746716,14.287055,https://www.openstreetmap.org/?mlat=59.7467163&mlon=14.2870549#map=18/59.7467163/14.2870549,None,None,LineString,121.269699,0.0,"{""OBJECTID"": 12351840, ""GEOMETRIKVALITET"": "">20-50 meter"", ""LED_ID"": ""30207434"", ""LEDNAMN"": ""Led Högbergsfältet"", ""B..."
6,12351838,30207434,Led Högbergsfältet,Vandringsled,Barmarksled,59.742346,14.279421,https://www.openstreetmap.org/?mlat=59.7423465&mlon=14.2794208#map=18/59.7423465/14.2794208,None,None,LineString,185.141262,0.0,"{""OBJECTID"": 12351838, ""GEOMETRIKVALITET"": "">20-50 meter"", ""LED_ID"": ""30207434"", ""LEDNAMN"": ""Led Högbergsfältet"", ""B..."
7,12351837,30207434,Led Högbergsfältet,Vandringsled,Barmarksled,59.744754,14.287799,https://www.openstreetmap.org/?mlat=59.7447538&mlon=14.2877986#map=18/59.7447538/14.2877986,None,None,LineString,353.709159,0.0,"{""OBJECTID"": 12351837, ""GEOMETRIKVALITET"": "">20-50 meter"", ""LED_ID"": ""30207434"", ""LEDNAMN"": ""Led Högbergsfältet"", ""B..."
8,12351836,30207434,Led Högbergsfältet,Vandringsled,Barmarksled,59.745748,14.287850,https://www.openstreetmap.org/?mlat=59.7457483&mlon=14.2878495#map=18/59.7457483/14.2878495,None,None,LineString,125.945522,0.0,"{""OBJECTID"": 12351836, ""GEOMETRIKVALITET"": "">20-50 meter"", ""LED_ID"": ""30207434"", ""LEDNAMN"": ""Led Högbergsfältet"", ""B..."
9,12355102,30207434,Led Högbergsfältet,Vandringsled,Barmarksled,59.742290,14.287195,https://www.openstreetmap.org/?mlat=59.7422902&mlon=14.2871955#map=18/59.7422902/14.2871955,None,None,LineString,52.729856,0.0,"{""OBJECTID"": 12355102, ""GEOMETRIKVALITET"": "">20-50 meter"", ""LED_ID"": ""30207434"", ""LEDNAMN"": ""Led Högbergsfältet"", ""B..."



ANORDNINGAR


,OBJECTID,latitude_wgs84,longitude_wgs84,osm_link,wikidata_id,wikidata_link,geometry_type,length_m,area_m2,metadata_json
0,33776721,55.664043,14.277837,https://www.openstreetmap.org/?mlat=55.6640430&mlon=14.2778371#map=18/55.6640430/14.2778371,None,None,Point,0.0,0.0,"{""OBJECTID"": 33776721, ""ANORDNING_ID"": ""30335917"", ""ANORDNINGNAMN"": null, ""TYP"": ""Fyr"", ""UNDERTYP"": ""Fyr"", ""BESKRIVN..."
1,33788626,56.300967,12.451604,https://www.openstreetmap.org/?mlat=56.3009668&mlon=12.4516037#map=18/56.3009668/12.4516037,None,None,Point,0.0,0.0,"{""OBJECTID"": 33788626, ""ANORDNING_ID"": ""30474194"", ""ANORDNINGNAMN"": null, ""TYP"": ""Fyr"", ""UNDERTYP"": ""Fyr"", ""BESKRIVN..."
2,33781407,56.450637,12.542535,https://www.openstreetmap.org/?mlat=56.4506368&mlon=12.5425355#map=18/56.4506368/12.5425355,None,None,Point,0.0,0.0,"{""OBJECTID"": 33781407, ""ANORDNING_ID"": ""30467932"", ""ANORDNINGNAMN"": null, ""TYP"": ""Fyr"", ""UNDERTYP"": ""Fyr"", ""BESKRIVN..."
3,33778187,59.297888,13.296044,https://www.openstreetmap.org/?mlat=59.2978885&mlon=13.2960436#map=18/59.2978885/13.2960436,None,None,Point,0.0,0.0,"{""OBJECTID"": 33778187, ""ANORDNING_ID"": ""30208827"", ""ANORDNINGNAMN"": ""Vägbom med kodlås"", ""TYP"": ""Vägbom"", ""UNDERTYP""..."
4,33785368,55.602717,14.198595,https://www.openstreetmap.org/?mlat=55.6027172&mlon=14.1985954#map=18/55.6027172/14.1985954,None,None,Point,0.0,0.0,"{""OBJECTID"": 33785368, ""ANORDNING_ID"": ""30570893"", ""ANORDNINGNAMN"": null, ""TYP"": ""Vägbom"", ""UNDERTYP"": ""Vägbom"", ""BE..."
5,33772787,58.271699,16.261009,https://www.openstreetmap.org/?mlat=58.2716993&mlon=16.2610092#map=18/58.2716993/16.2610092,None,None,Point,0.0,0.0,"{""OBJECTID"": 33772787, ""ANORDNING_ID"": ""30562476"", ""ANORDNINGNAMN"": null, ""TYP"": ""Vägbom"", ""UNDERTYP"": ""Vägbom"", ""BE..."
6,33788228,58.673347,16.120040,https://www.openstreetmap.org/?mlat=58.6733472&mlon=16.1200403#map=18/58.6733472/16.1200403,None,None,Point,0.0,0.0,"{""OBJECTID"": 33788228, ""ANORDNING_ID"": ""30562364"", ""ANORDNINGNAMN"": null, ""TYP"": ""Vägbom"", ""UNDERTYP"": ""Vägbom"", ""BE..."
7,33789997,58.810788,17.344025,https://www.openstreetmap.org/?mlat=58.8107876&mlon=17.3440252#map=18/58.8107876/17.3440252,None,None,Point,0.0,0.0,"{""OBJECTID"": 33789997, ""ANORDNING_ID"": ""30557502"", ""ANORDNINGNAMN"": null, ""TYP"": ""Vägbom"", ""UNDERTYP"": ""Vägbom"", ""BE..."
8,33781659,60.856267,12.850427,https://www.openstreetmap.org/?mlat=60.8562669&mlon=12.8504266#map=18/60.8562669/12.8504266,None,None,Point,0.0,0.0,"{""OBJECTID"": 33781659, ""ANORDNING_ID"": ""30540649"", ""ANORDNINGNAMN"": ""Stängd vägbom Vålhallberget"", ""TYP"": ""Vägbom"", ..."
9,33785038,60.301296,13.273160,https://www.openstreetmap.org/?mlat=60.3012956&mlon=13.2731602#map=18/60.3012956/13.2731602,None,None,Point,0.0,0.0,"{""OBJECTID"": 33785038, ""ANORDNING_ID"": ""30540645"", ""ANORDNINGNAMN"": ""Vägbom Tijärnsskogen, Dörrfjället"", ""TYP"": ""Väg..."



PUBLICERINGSSTATUS


,latitude_wgs84,longitude_wgs84,osm_link,wikidata_id,wikidata_link,geometry_type,length_m,area_m2,metadata_json
0,56.963203,16.898328,https://www.openstreetmap.org/?mlat=56.9632034&mlon=16.8983281#map=18/56.9632034/16.8983281,None,None,Polygon,2848.316814,3.383315e+05,"{""NVRID"": ""2011415"", ""NAMN"": ""Kåreholm och Kartarna"", ""DATA_FRILUFTSLIV_PUBLICERAT"": ""Nej""}"
1,57.504141,14.171565,https://www.openstreetmap.org/?mlat=57.5041406&mlon=14.1715654#map=18/57.5041406/14.1715654,None,None,Polygon,4749.965846,3.181713e+05,"{""NVRID"": ""2014384"", ""NAMN"": ""Östermoskogen"", ""DATA_FRILUFTSLIV_PUBLICERAT"": ""Ja""}"
2,56.697725,16.504507,https://www.openstreetmap.org/?mlat=56.6977249&mlon=16.5045067#map=18/56.6977249/16.5045067,None,None,Polygon,3.140344,7.841428e-01,"{""NVRID"": ""2011931"", ""NAMN"": ""Idegranar"", ""DATA_FRILUFTSLIV_PUBLICERAT"": ""Nej""}"
3,64.744432,17.897177,https://www.openstreetmap.org/?mlat=64.7444324&mlon=17.8971772#map=18/64.7444324/17.8971772,None,None,Polygon,3238.313061,2.024649e+05,"{""NVRID"": ""2201506"", ""NAMN"": ""Svartliden"", ""DATA_FRILUFTSLIV_PUBLICERAT"": ""Nej""}"
4,56.047532,15.787041,https://www.openstreetmap.org/?mlat=56.0475323&mlon=15.7870412#map=18/56.0475323/15.7870412,None,None,Polygon,1992.505204,1.748176e+05,"{""NVRID"": ""2003657"", ""NAMN"": ""Kuggaskär"", ""DATA_FRILUFTSLIV_PUBLICERAT"": ""Nej""}"
5,58.070753,15.360574,https://www.openstreetmap.org/?mlat=58.0707528&mlon=15.3605737#map=18/58.0707528/15.3605737,None,None,Polygon,5560.546863,6.691207e+05,"{""NVRID"": ""2052762"", ""NAMN"": ""Månhult"", ""DATA_FRILUFTSLIV_PUBLICERAT"": ""Ja""}"
6,61.405314,14.034604,https://www.openstreetmap.org/?mlat=61.4053140&mlon=14.0346042#map=18/61.4053140/14.0346042,None,None,Polygon,961.802408,5.330636e+04,"{""NVRID"": ""2002477"", ""NAMN"": ""Prästskogsstugan"", ""DATA_FRILUFTSLIV_PUBLICERAT"": ""Nej""}"
7,63.325494,14.226537,https://www.openstreetmap.org/?mlat=63.3254939&mlon=14.2265370#map=18/63.3254939/14.2265370,None,None,Polygon,3256.242964,3.649085e+05,"{""NVRID"": ""2000892"", ""NAMN"": ""Önet"", ""DATA_FRILUFTSLIV_PUBLICERAT"": ""Ja""}"
8,58.846803,15.617311,https://www.openstreetmap.org/?mlat=58.8468031&mlon=15.6173110#map=18/58.8468031/15.6173110,None,None,Polygon,17747.430224,3.655396e+06,"{""NVRID"": ""2055946"", ""NAMN"": ""Stora mossen"", ""DATA_FRILUFTSLIV_PUBLICERAT"": ""Ja""}"
9,60.833990,12.638593,https://www.openstreetmap.org/?mlat=60.8339901&mlon=12.6385930#map=18/60.8339901/12.6385930,None,None,Polygon,3704.807513,4.072129e+05,"{""NVRID"": ""2200010"", ""NAMN"": ""Lövberget"", ""DATA_FRILUFTSLIV_PUBLICERAT"": ""Nej""}"



STATLIGA_LEDER


,OBJECTID,latitude_wgs84,longitude_wgs84,osm_link,wikidata_id,wikidata_link,geometry_type,length_m,area_m2,metadata_json
0,12348588,63.076100,12.815285,https://www.openstreetmap.org/?mlat=63.0760999&mlon=12.8152845#map=18/63.0760999/12.8152845,None,None,LineString,985.534369,0.0,"{""OBJECTID"": 12348588, ""STATLIGLED"": ""Z 59"", ""STATLIGLEDNAMN"": ""Vålådalen - Vålåstugorna (justerad)"", ""SHAPE.LEN"": 9..."
1,12348587,63.057487,12.799666,https://www.openstreetmap.org/?mlat=63.0574868&mlon=12.7996660#map=18/63.0574868/12.7996660,None,None,LineString,3766.922884,0.0,"{""OBJECTID"": 12348587, ""STATLIGLED"": ""Z 59"", ""STATLIGLEDNAMN"": ""Vålådalen - Vålåstugorna (justerad)"", ""SHAPE.LEN"": 3..."
2,12348586,63.143722,12.906417,https://www.openstreetmap.org/?mlat=63.1437223&mlon=12.9064169#map=18/63.1437223/12.9064169,None,None,LineString,2391.088990,0.0,"{""OBJECTID"": 12348586, ""STATLIGLED"": ""Z 59"", ""STATLIGLEDNAMN"": ""Vålådalen - Vålåstugorna (justerad)"", ""SHAPE.LEN"": 2..."
3,12348585,63.147378,12.944144,https://www.openstreetmap.org/?mlat=63.1473780&mlon=12.9441443#map=18/63.1473780/12.9441443,None,None,LineString,2523.903544,0.0,"{""OBJECTID"": 12348585, ""STATLIGLED"": ""Z 59"", ""STATLIGLEDNAMN"": ""Vålådalen - Vålåstugorna (justerad)"", ""SHAPE.LEN"": 2..."
4,12348553,63.108612,12.835620,https://www.openstreetmap.org/?mlat=63.1086121&mlon=12.8356204#map=18/63.1086121/12.8356204,None,None,LineString,8137.139796,0.0,"{""OBJECTID"": 12348553, ""STATLIGLED"": ""Z 59"", ""STATLIGLEDNAMN"": ""Vålådalen - Vålåstugorna (justerad)"", ""SHAPE.LEN"": 8..."
5,12348552,63.071994,12.812557,https://www.openstreetmap.org/?mlat=63.0719940&mlon=12.8125573#map=18/63.0719940/12.8125573,None,None,LineString,192.041971,0.0,"{""OBJECTID"": 12348552, ""STATLIGLED"": ""Z 59"", ""STATLIGLEDNAMN"": ""Vålådalen - Vålåstugorna (justerad)"", ""SHAPE.LEN"": 1..."
6,12348551,63.133399,12.881880,https://www.openstreetmap.org/?mlat=63.1333991&mlon=12.8818804#map=18/63.1333991/12.8818804,None,None,LineString,1300.291774,0.0,"{""OBJECTID"": 12348551, ""STATLIGLED"": ""Z 59"", ""STATLIGLEDNAMN"": ""Vålådalen - Vålåstugorna (justerad)"", ""SHAPE.LEN"": 1..."
7,12344819,63.135791,13.687877,https://www.openstreetmap.org/?mlat=63.1357912&mlon=13.6878772#map=18/63.1357912/13.6878772,None,None,LineString,3.130011,0.0,"{""OBJECTID"": 12344819, ""STATLIGLED"": ""Z 70"", ""STATLIGLEDNAMN"": ""Spjätten - Grottärn"", ""SHAPE.LEN"": 3.13001086260096}"
8,12344818,63.111568,13.716483,https://www.openstreetmap.org/?mlat=63.1115683&mlon=13.7164825#map=18/63.1115683/13.7164825,None,None,LineString,350.629881,0.0,"{""OBJECTID"": 12344818, ""STATLIGLED"": ""Z 70"", ""STATLIGLEDNAMN"": ""Spjätten - Grottärn"", ""SHAPE.LEN"": 350.629880667136}"
9,12344817,63.136029,13.687674,https://www.openstreetmap.org/?mlat=63.1360289&mlon=13.6876745#map=18/63.1360289/13.6876745,None,None,LineString,18.082741,0.0,"{""OBJECTID"": 12344817, ""STATLIGLED"": ""Z 70"", ""STATLIGLEDNAMN"": ""Spjätten - Grottärn"", ""SHAPE.LEN"": 18.0827410012973}"


## 14. Export – analysresultat

Resultaten sparas som CSV/Excel. GeoJSON med WGS84 kan också exporteras om man vill fortsätta i QGIS, uMap, OSM eller andra GIS-verktyg.


In [13]:
# 15. Exportera resultat

OUTPUT_DIR = Path("naturvardsverket_friluftsliv_output")
OUTPUT_DIR.mkdir(exist_ok=True)

for name, df in analysed.items():
    # Tabellversion utan geometri för enkel CSV/Excel.
    csv_df = pd.DataFrame(df.drop(columns="geometry"))
    csv_path = OUTPUT_DIR / f"{name}_metadata_wgs84.csv"
    csv_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    # GeoJSON i WGS84 – behåller alla ursprungliga attribut + analysfält.
    geo = df.to_crs(TARGET_CRS)
    geo_path = OUTPUT_DIR / f"{name}_wgs84.geojson"
    geo.to_file(geo_path, driver="GeoJSON")

    print(f"{name}:")
    print(f"  CSV:     {csv_path}")
    print(f"  GeoJSON: {geo_path}")

coverage.to_csv(
    OUTPUT_DIR / "coverage_osm_wikidata.csv",
    index=False,
    encoding="utf-8-sig"
)

identifier_report.to_csv(
    OUTPUT_DIR / "identifier_report.csv",
    index=False,
    encoding="utf-8-sig"
)

for name, summary in type_summary.items():
    summary.to_csv(
        OUTPUT_DIR / f"{name}_type_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

print("\nKlart.")


leder:
  CSV:     naturvardsverket_friluftsliv_output/leder_metadata_wgs84.csv
  GeoJSON: naturvardsverket_friluftsliv_output/leder_wgs84.geojson
anordningar:
  CSV:     naturvardsverket_friluftsliv_output/anordningar_metadata_wgs84.csv
  GeoJSON: naturvardsverket_friluftsliv_output/anordningar_wgs84.geojson
publiceringsstatus:
  CSV:     naturvardsverket_friluftsliv_output/publiceringsstatus_metadata_wgs84.csv
  GeoJSON: naturvardsverket_friluftsliv_output/publiceringsstatus_wgs84.geojson
statliga_leder:
  CSV:     naturvardsverket_friluftsliv_output/statliga_leder_metadata_wgs84.csv
  GeoJSON: naturvardsverket_friluftsliv_output/statliga_leder_wgs84.geojson

Klart.


## 16. Viktig tolkning av OSM-länkar

`osm_link` har två nivåer:

1. **Riktig OSM-objektlänk** om datat innehåller OSM-ID och objekttyp.
2. **Koordinatlänk till OSM** annars.

Det senare betyder **inte** att objektet finns i OSM. Det är bara en geografisk länk till platsen.

På motsvarande sätt är `wikidata_link` bara ifylld när ett Wikidata-QID faktiskt hittas i attributen.

Det gör att vi kan skilja på:
- *”Naturvårdsverket har en explicit Linked Data-koppling”*
- *”vi kan öppna samma plats i OSM”*.

Det är viktigt för en FAIR/Linked Data-analys.
